# Reflectivity diffraction-limit probe (q-truncation)

Classical length scale of the measured reflectivity: $d = \pi / q_{\mathrm{cut}}$.
At each cut we re-polish the locked multi-energy bookended `logY` model on $q \le q_{\mathrm{cut}}$, then score the held-out high-$q$ fringe (almost entirely 283.7 eV).

Scan window: $q_{\mathrm{hi}} = q_{\max}(283.7)$ down to $q_{\mathrm{lo}} = \min q_{\max}$ of the other resonant energies.

Runner: `_bookended_q_truncation_diffraction.py` → `@models/fitting_results/bookended_q_truncation_diffraction/`.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils import models_fitting_results

OUT = models_fitting_results / "bookended_q_truncation_diffraction"
df = pd.read_parquet(OUT / "scan.parquet").sort_values("q_cut", ascending=False).reset_index(drop=True)
meta = json.loads((OUT / "summary.json").read_text())
d_full = float(meta["d_full"])
q_hi = float(meta["q_hi"])
q_lo = float(meta["q_lo"])
df

## Soft curves vs reflectivity diffraction length

Train = $q \le q_{\mathrm{cut}}$ (all energies). Holdout = $q > q_{\mathrm{cut}}$ (283.7 eV fringe). Vertical line marks the full-data reflectivity diffraction limit $\pi / q_{\max}$.

In [ ]:
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.linewidth": 0.8,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

plot_df = df.sort_values("d_pi_over_q")
hold = plot_df["n_holdout"] > 0

fig, axes = plt.subplots(2, 1, figsize=(5.4, 5.2), sharex=True, gridspec_kw={"hspace": 0.14})

ax = axes[0]
ax.plot(plot_df["d_pi_over_q"], plot_df["redchi_train"], "o-", ms=3.5, color="0.2", label="train")
ax.plot(
    plot_df.loc[hold, "d_pi_over_q"],
    plot_df.loc[hold, "redchi_holdout"],
    "s-",
    ms=3.5,
    color="#C0392B",
    label="holdout",
)
ax.axvline(
    d_full,
    color="0.45",
    ls=":",
    lw=0.9,
    label=rf"reflectivity $d_{{\mathrm{{lim}}}}=\pi/q_{{\max}}$ = {d_full:.2f} Å",
)
ax.set_ylabel(r"train red.$\chi^2$ / holdout $\chi^2/n$")
ax.legend(frameon=False, loc="best", fontsize=7.5)

ax = axes[1]
ax.plot(plot_df["d_pi_over_q"], plot_df["rms_logR_train"], "o-", ms=3.5, color="0.2", label="train")
ax.plot(
    plot_df.loc[hold, "d_pi_over_q"],
    plot_df.loc[hold, "rms_logR_holdout"],
    "s-",
    ms=3.5,
    color="#C0392B",
    label="holdout",
)
ax.axvline(d_full, color="0.45", ls=":", lw=0.9)
ax.set_xlabel(r"reflectivity diffraction length $d=\pi/q_{\mathrm{cut}}$ (Å)")
ax.set_ylabel(r"RMS($\log_{10} R$)")
ax.legend(frameon=False, loc="best", fontsize=7.5)

ax_top = axes[0].secondary_xaxis(
    "top",
    functions=(
        lambda d: np.pi / np.asarray(d, dtype=float),
        lambda q: np.pi / np.asarray(q, dtype=float),
    ),
)
ax_top.set_xlabel(r"$q_{\mathrm{cut}}$ (Å$^{-1}$)")

fig.suptitle(
    "Holdout generalization vs reflectivity diffraction limit",
    fontsize=10,
)
fig.savefig(OUT / "soft_curves.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / "soft_curves.pdf", bbox_inches="tight")
fig

In [ ]:
fig2, ax = plt.subplots(figsize=(5.4, 2.7))
ax.plot(plot_df["d_pi_over_q"], plot_df["tau_vac"], "o-", ms=3.5, color="0.2", label=r"$\tau_{\mathrm{vac}}$ (Å)")
ax.plot(
    plot_df["d_pi_over_q"],
    np.rad2deg(plot_df["alpha_vac"]),
    "s-",
    ms=3.5,
    color="#C0392B",
    label=r"$\alpha_{\mathrm{vac}}$ (deg)",
)
ax.axvline(d_full, color="0.45", ls=":", lw=0.9, label=rf"$d_{{\mathrm{{lim}}}}$ = {d_full:.2f} Å")
ax.set_xlabel(r"reflectivity diffraction length $d=\pi/q_{\mathrm{cut}}$ (Å)")
ax.set_ylabel("parameter")
ax.legend(frameon=False, loc="best", fontsize=7.5)
fig2.savefig(OUT / "params_vs_d.png", dpi=300, bbox_inches="tight")
fig2.savefig(OUT / "params_vs_d.pdf", bbox_inches="tight")
fig2

## Reflectivity cascade over all $q_{\mathrm{cut}}$

Full cascade of truncated fits on 283.7 eV $\log_{10} R$. Curves and dashed vertical lines are colored by $q_{\mathrm{cut}}$ (colorbar). Holdout for each fit is $q > q_{\mathrm{cut}}$. Prefers cached `reflectivity_q_cuts_curves.npz` if present.

In [ ]:
CACHE = OUT / "reflectivity_q_cuts_curves.npz"
REBUILD = False

q_cuts = [float(q) for q in df["q_cut"].to_numpy()]
curve_by_cut: dict[float, dict] = {}

if CACHE.exists() and not REBUILD:
    z = np.load(CACHE, allow_pickle=True)
    q_cuts = [float(q) for q in np.asarray(z["q_cuts"], dtype=float)]
    for i, q_cut in enumerate(q_cuts):
        curve_by_cut[q_cut] = {
            "q": np.asarray(z[f"q_{i}"], dtype=float),
            "logR": np.asarray(z[f"logR_{i}"], dtype=float),
            "logR_err": np.asarray(z[f"logR_err_{i}"], dtype=float),
            "logR_fit": np.asarray(z[f"logR_fit_{i}"], dtype=float),
            "pol": np.asarray(z[f"pol_{i}"]),
        }
    print(f"loaded cache {CACHE.name}: {len(q_cuts)} cuts")
else:
    _HERE = Path("notebooks/fitting")
    if not (_HERE / "_bookended_q_truncation_diffraction.py").exists():
        _HERE = Path(".")
    _path = (_HERE / "_bookended_q_truncation_diffraction.py").resolve()
    _spec = importlib.util.spec_from_file_location("_q_trunc_scan", _path)
    assert _spec is not None and _spec.loader is not None
    scan = importlib.util.module_from_spec(_spec)
    sys.modules[_spec.name] = scan
    _spec.loader.exec_module(scan)

    summary = scan.load_polish_summary()
    objectives = scan.cv.seed_objectives()
    obj_map = {float(e): objectives[float(e)] for e in scan.ENERGIES}
    for i, q_cut in enumerate(q_cuts):
        print(f"[{i + 1}/{len(q_cuts)}] fitting q_cut={q_cut:.6f} ...", flush=True)
        _row, train = scan.fit_cut_objective(obj_map, summary, q_cut)
        curve_by_cut[q_cut] = scan.full_q_curves(obj_map, train, energy=283.7)
    np.savez_compressed(
        CACHE,
        q_cuts=np.asarray(q_cuts, dtype=float),
        **{f"q_{i}": curve_by_cut[q]["q"] for i, q in enumerate(q_cuts)},
        **{f"logR_{i}": curve_by_cut[q]["logR"] for i, q in enumerate(q_cuts)},
        **{f"logR_err_{i}": curve_by_cut[q]["logR_err"] for i, q in enumerate(q_cuts)},
        **{f"logR_fit_{i}": curve_by_cut[q]["logR_fit"] for i, q in enumerate(q_cuts)},
        **{f"pol_{i}": curve_by_cut[q]["pol"] for i, q in enumerate(q_cuts)},
    )
    print(f"wrote {CACHE}")

cmap = plt.cm.viridis(np.linspace(0.05, 0.95, len(q_cuts)))
print("cascade cuts:", len(q_cuts))

In [ ]:
fig3, axes = plt.subplots(1, 2, figsize=(7.6, 3.4), sharey=True, gridspec_kw={"wspace": 0.08})
DATA_KW = dict(
    fmt="o",
    ms=1.8,
    lw=0.35,
    color="0.45",
    ecolor="0.75",
    elinewidth=0.3,
    capsize=0,
    zorder=1,
    rasterized=True,
)

for col, pol_name in enumerate(("s", "p")):
    ax = axes[col]
    ref = curve_by_cut[q_cuts[0]]
    m = ref["pol"] == pol_name
    order = np.argsort(ref["q"][m])
    qq = ref["q"][m][order]
    ax.errorbar(qq, ref["logR"][m][order], yerr=ref["logR_err"][m][order], **DATA_KW)

    for q_cut, color in zip(q_cuts, cmap, strict=True):
        cur = curve_by_cut[q_cut]
        m = cur["pol"] == pol_name
        order = np.argsort(cur["q"][m])
        qq = cur["q"][m][order]
        yf = cur["logR_fit"][m][order]
        ax.plot(qq, yf, color=color, lw=0.95, zorder=2, alpha=0.95)
        ax.axvline(q_cut, color=color, ls="--", lw=0.55, alpha=0.55, zorder=0)

    ax.axvline(q_lo, color="0.15", ls=":", lw=1.0, alpha=0.9, zorder=3)
    ax.axvline(q_hi, color="0.15", ls="-", lw=0.7, alpha=0.5, zorder=3)
    ax.set_xlabel(r"$q$ (Å$^{-1}$)")
    ax.set_title(f"283.7 eV  {pol_name}", fontsize=9)
    if col == 0:
        ax.set_ylabel(r"$\log_{10} R$")

sm = plt.cm.ScalarMappable(
    cmap=plt.cm.viridis,
    norm=plt.Normalize(vmin=min(q_cuts), vmax=max(q_cuts)),
)
sm.set_array([])
cbar = fig3.colorbar(sm, ax=axes, fraction=0.035, pad=0.02)
cbar.set_label(r"$q_{\mathrm{cut}}$ (Å$^{-1}$)", fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig3.suptitle(
    "Cascade of truncated fits on full 283.7 eV reflectivity\n"
    r"dashed = each $q_{\mathrm{cut}}$; holdout is $q > q_{\mathrm{cut}}$; "
    rf"$q_{{\mathrm{{lo}}}}$={q_lo:.3f} (dotted), $q_{{\mathrm{{hi}}}}$={q_hi:.3f}",
    fontsize=9.5,
    y=1.08,
)
fig3.savefig(OUT / "reflectivity_q_cuts.png", dpi=300, bbox_inches="tight")
fig3.savefig(OUT / "reflectivity_q_cuts.pdf", bbox_inches="tight")
fig3

## Interpretation

Filled from the finished 21-point scan in the next cell.

In [ ]:
hold_df = df[df["n_holdout"] > 0].copy()
best_hold = hold_df.loc[hold_df["rms_logR_holdout"].idxmin()]
worst_hold = hold_df.loc[hold_df["rms_logR_holdout"].idxmax()]
full = df.iloc[0]

print(f"Full-data reflectivity diffraction limit: d_lim = pi/q_hi = {d_full:.3f} A")
print(f"Scan end (other resonances' q_max): q_lo = {q_lo:.4f}  ->  d = {np.pi/q_lo:.2f} A")
print()
print(
    f"Full-q re-polish: redchi_train={full['redchi_train']:.3f}, "
    f"tau_vac={full['tau_vac']:.2f} A (seed polish had tau_vac=32 A; hit {full['tau_vac']:.1f})"
)
print(
    f"Best holdout RMS(logR): {best_hold['rms_logR_holdout']:.3f} "
    f"at q_cut={best_hold['q_cut']:.4f} (d={best_hold['d_pi_over_q']:.2f} A, "
    f"factor={best_hold['d_pi_over_q']/d_full:.2f} x d_lim)"
)
print(
    f"Worst holdout RMS(logR): {worst_hold['rms_logR_holdout']:.3f} "
    f"at q_cut={worst_hold['q_cut']:.4f} (d={worst_hold['d_pi_over_q']:.2f} A, "
    f"factor={worst_hold['d_pi_over_q']/d_full:.2f} x d_lim)"
)
print(
    f"Holdout chi2/n range: {hold_df['redchi_holdout'].min():.2f} .. "
    f"{hold_df['redchi_holdout'].max():.2f}"
)
print(f"tau_vac at upper bound 55 A for {(df['tau_vac'] >= 54.99).sum()}/{len(df)} cuts")
print()
print(
    "Takeaway: holdout RMS stays ~2.1–2.7 across the whole window while d grows from "
    f"~{hold_df['d_pi_over_q'].min():.1f} to {hold_df['d_pi_over_q'].max():.1f} A "
    f"({hold_df['d_pi_over_q'].min()/d_full:.1f}–{hold_df['d_pi_over_q'].max()/d_full:.1f} x the "
    "full-data reflectivity diffraction limit). There is no sharp cliff: high-q 283.7 eV "
    "remains predictable from truncated fits, but tau_vac saturates at its bound as soon as "
    "any high-q fringe is removed — the surface decay length is only weakly constrained "
    "without that fringe."
)